# Python 기초 4 · 직접 풀어보는 연습문제 정답지

이 노트북은 `Python_Basic_4_Data_Classification_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 실습에 필요한 Pandas 데이터셋을 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np

sample_data = pd.DataFrame({
    "user_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "region": ["Seoul", "Busan", "Seoul", "Daegu", "Seoul", "Incheon", "Busan", "Seoul"],
    "satisfaction": ["Satisfied", "Very Satisfied", "Neutral", "Dissatisfied", "Satisfied", "Neutral", "Very Satisfied", "Satisfied"],
    "cart_items": [2, 5, 1, 0, 4, 2, 7, 3],
    "order_amount": [54000.0, 182000.0, 24000.0, 0.0, 129000.0, 45000.0, 310000.0, 88000.0],
    "churn": [0, 0, 1, 1, 0, 0, 0, 0] # 이탈 여부: 1=이탈, 0=유지
})


## 문제 1. 데이터셋 컬럼별 변수 척도(명목/순서/이산/연속) 판정 [개념 적용]

`sample_data` 데이터프레임에는 총 6개의 컬럼이 존재합니다:
`['user_id', 'region', 'satisfaction', 'cart_items', 'order_amount', 'churn']`

각 컬럼에 대해 데이터 분석가의 기준에 따라 변수 유형을 분류한 딕셔너리 `variable_types`를 작성하여 출력하세요.
- 분류 기준값: `"명목형"`, `"순서형"`, `"수치-이산형"`, `"수치-연속형"`
- 출력 형식 예: `{'region': '명목형', ...}`


In [ ]:
# 문제 1 풀이
variable_types = {
    "user_id": "명목형",         # 단순 고유 식별자(ID)
    "region": "명목형",          # 순서가 없는 범주
    "satisfaction": "순서형",    # 만족도 위계 존재 (Dissatisfied < Neutral < Satisfied < Very Satisfied)
    "cart_items": "수치-이산형",   # 개수를 세는 정수 단위
    "order_amount": "수치-연속형", # 화폐 단위, 연속적인 금액
    "churn": "명목형"            # 0과 1로 코딩된 이진 범주(이탈 여부)
}

print("=== 컬럼별 변수 분류 결과 ===")
for col, vtype in variable_types.items():
    print(f"컬럼: {col:<15} | 척도: {vtype}")


**결과 해석 / 확인한 점:**  
1. **식별자 및 이진 변수의 척도:** `user_id`나 `churn`은 숫자로 저장되어 있더라도 사칙연산이 무의미한 단순 식별 및 이진 분류 레이블이므로 명목형(Nominal)으로 분류하는 것이 분석학적으로 타당합니다.
2. **연속형과 이산형의 구분:** `cart_items`는 셀 수 있는 정수 개수(Count)이므로 이산형, `order_amount`는 소수점 이하 측정이 가능한 금액이므로 연속형입니다.


## 문제 2. object 문자열 컬럼의 Category 타입 변환 및 메모리 비교 [타입 최적화]

대용량 데이터에서 범주형 문자열(`object`)을 `category` 타입으로 변환하면 메모리를 대폭 절약할 수 있습니다.
1. `sample_data['region']`의 현재 데이터 타입과 메모리 사용량(`memory_usage()`)을 확인하세요.
2. 해당 컬럼을 `.astype('category')`로 변환하세요.
3. 변환 후의 dtype과 고유 범주 목록(`.cat.categories`)을 출력하세요.


In [ ]:
# 문제 2 풀이
# 1. 변환 전 확인
before_dtype = sample_data["region"].dtype
before_mem = sample_data["region"].memory_usage(deep=True)
print(f"변환 전 region dtype: {before_dtype}, 메모리: {before_mem} bytes")

# 2. category 타입 변환
sample_data["region"] = sample_data["region"].astype("category")

# 3. 변환 후 확인
after_dtype = sample_data["region"].dtype
after_mem = sample_data["region"].memory_usage(deep=True)
print(f"변환 후 region dtype: {after_dtype}, 메모리: {after_mem} bytes")
print("고유 범주 카테고리:", list(sample_data["region"].cat.categories))


**결과 해석 / 확인한 점:**  
1. **정수 인코딩 기반 최적화:** `category` 타입은 중복되는 문자열을 내부적으로 1바이트 정수 코드(Code)와 카테고리 룩업 테이블로 관리하므로, 반복 빈도가 높은 대용량 데이터셋에서 RAM 점유율을 50~80% 이상 절감할 수 있습니다.
2. **범주 목록 자동 색인:** `Seoul`, `Busan`, `Daegu`, `Incheon` 4개의 고유 범주가 인덱싱되었습니다.


## 문제 3. 순서형(Ordinal) 범주에 명시적 순서 부여 [순서형 인코딩]

`satisfaction` 컬럼은 다음과 같은 명확한 만족도 위계를 가집니다:
- `Dissatisfied` (불만족) < `Neutral` (보통) < `Satisfied` (만족) < `Very Satisfied` (매우 만족)

`pd.CategoricalDtype`을 사용하여:
1. 위 4단계 순서를 갖는 순서형 카테고리 타입(`ordered_satisfaction`)을 정의하세요.
2. `sample_data['satisfaction']`에 적용하여 크기 비교 연산(`df['satisfaction'] >= 'Satisfied'`)이 작동하도록 만드세요.
3. '만족(Satisfied)' 이상인 고객 수만 필터링하여 출력하세요.


In [ ]:
# 문제 3 풀이
from pandas.api.types import CategoricalDtype

# 1. 명시적 순서를 갖는 범주형 타입 정의
satisfaction_order = ["Dissatisfied", "Neutral", "Satisfied", "Very Satisfied"]
sat_dtype = CategoricalDtype(categories=satisfaction_order, ordered=True)

# 2. 컬럼 적용
sample_data["satisfaction"] = sample_data["satisfaction"].astype(sat_dtype)
print("적용된 순서형 카테고리 정보:", sample_data["satisfaction"].dtype)

# 3. '만족' 이상인 고객 필터링
satisfied_mask = sample_data["satisfaction"] >= "Satisfied"
satisfied_customers = sample_data[satisfied_mask]

print(f"\n만족(Satisfied) 이상인 고객 수: {len(satisfied_customers)}명 (총 {len(sample_data)}명 중)")
print(satisfied_customers[["user_id", "satisfaction", "order_amount"]])


**결과 해석 / 확인한 점:**  
1. **순서형 크기 비교(Inequality Comparison):** `ordered=True` 옵션을 지정하면 문자열 상태에서도 부등호(`>=`)를 통한 논리 인덱싱이 가능해져 설문 조사 및 등급 분석이 직관화됩니다.
2. **필터링 결과:** 총 8명 중 Satisfied 3명, Very Satisfied 2명으로 총 5명의 만족 고객이 필터링되었습니다.


## 문제 4. 범주형 변수의 도수분포표(빈도 및 백분율) 생성 [기초 통계 탐색]

범주형 변수의 대표적인 탐색 방법은 도수분포표(Frequency Table)입니다.
`sample_data['region']` 컬럼에 대해:
1. 각 지역별 빈도수(`count`)를 구하세요 (`value_counts()`).
2. 각 지역별 백분율 비율(`percentage`, %)을 소수점 첫째 자리까지 구하세요 (`normalize=True`).
3. 두 지표를 묶어 하나의 데이터프레임 `freq_table`로 만들고 출력하세요.


In [ ]:
# 문제 4 풀이
counts = sample_data["region"].value_counts()
pcts = sample_data["region"].value_counts(normalize=True) * 100

freq_table = pd.DataFrame({
    "빈도수(명)": counts,
    "비율(%)": pcts.round(1)
})

print("=== 지역별 유입 도수분포표 ===")
print(freq_table)


**결과 해석 / 확인한 점:**  
1. **도수분포표의 분석 가치:** 서울 거주자가 4명(50.0%)으로 절반을 차지하며, 부산 2명(25.0%), 대구 및 인천 각 1명(12.5%)으로 수도권 및 대도시 중심의 유저 분포가 확인되었습니다.
2. **value_counts(normalize=True):** 데이터 건수와 상대적 점유율을 동시에 제시하는 것이 경영진 보고용 지표 작성의 표준입니다.


## 문제 5. 연속형 변수(order_amount)의 사분위수 기반 구간화(pd.qcut) [변수 변환]

연속형 수치 데이터인 구매 금액(`order_amount`)을 데이터 분석 모델의 범주형 특성으로 변환(Binning)하고자 합니다.
`pd.qcut()`을 사용하여:
1. `order_amount`를 동일 관측치 비율로 3개 구간(Low, Medium, High)으로 분할하여 `spend_level` 컬럼을 생성하세요.
2. 각 구간별 관측치 수와 구간 경계값을 확인하세요.


In [ ]:
# 문제 5 풀이
labels = ["Low", "Medium", "High"]
sample_data["spend_level"] = pd.qcut(sample_data["order_amount"], q=3, labels=labels)

print("=== 사분위수 구간화 결과 ===")
print(sample_data[["user_id", "order_amount", "spend_level"]])
print("\n각 레벨별 고객 수:")
print(sample_data["spend_level"].value_counts())


**결과 해석 / 확인한 점:**  
1. **qcut vs cut:** `pd.cut()`은 수치 범위를 균등하게 나눌 때 사용하고, `pd.qcut()`은 각 구간에 동일한 수의 데이터가 들어가도록 분위수(Quantile) 기준으로 나눕니다.
2. **연속형의 범주화 목적:** 비선형적 관계를 선형 모델에 반영하거나 고객을 직관적인 3단계 티어로 단순화할 때 매우 유용한 피처 엔지니어링 기법입니다.


## 문제 6. 머신러닝 모형 입력을 위한 독립변수 X와 종속변수 Y 분리 [피처 엔지니어링]

우리의 최종 목표는 고객의 특성(`region`, `satisfaction`, `cart_items`, `order_amount`)을 바탕으로 고객 이탈 여부(`churn`)를 예측하는 것입니다.
1. 고객 식별자인 `user_id`와 새로 생성된 파생변수 `spend_level`을 제외하세요.
2. 모델 학습에 사용할 피처 매트릭스 `X` (독립변수 데이터프레임)를 추출하세요.
3. 예측 대상인 타겟 벡터 `y` (`churn` 컬럼)를 분리하세요.
4. `X.shape`와 `y.shape`를 출력하여 차원을 검증하세요.


In [ ]:
# 문제 6 풀이
# 1. 제외할 컬럼 목록
drop_cols = ["user_id", "spend_level", "churn"]

# 2. 독립변수 매트릭스 X 추출
X = sample_data.drop(columns=drop_cols)

# 3. 종속변수 타겟 벡터 y 분리
y = sample_data["churn"]

print("독립변수 X 형태:", X.shape)
print("독립변수 컬럼 목록:", list(X.columns))
print("종속변수 y 형태:", y.shape)
print("\n독립변수 X 미리보기:")
print(X.head())


**결과 해석 / 확인한 점:**  
1. **X와 y의 차원 분리:** 독립변수 $X$는 8개 행과 4개 피처(region, satisfaction, cart_items, order_amount)를 갖는 2차원 DataFrame(`(8, 4)`)으로, 종속변수 $y$는 8개의 레이블을 갖는 1차원 Series(`(8,)`)로 성공적으로 분리되었습니다.
2. **데이터 누수(Leakage) 방지:** 고객 식별 번호(`user_id`)나 타겟 변수에서 파생된 변수를 사전에 제거함으로써 모델의 과적합(Overfitting)과 데이터 누수를 방지했습니다.
